<img src=../figures/Brown_logo.svg width=50%>

## Data-Driven Design & Analyses of Structures & Materials (3dasm)

## Lecture 19.2

### Elvis Aguero | <a href = "mailto: elvis_alexander_aguero_vera@brown.edu">elvis_alexander_aguero_vera@brown.edu</a>  | PhD candidate

### Miguel A. Bessa | <a href = "mailto: miguel_bessa@brown.edu">miguel_bessa@brown.edu</a>  | Associate Professor

**What:** A lecture of the "3dasm" course

**Where:** This notebook comes from this [repository](https://github.com/bessagroup/3dasm_course)

**Reference for this lecture:** Bessa, M. A., Glowacki, P., & Houlder, M. (2019). *Bayesian
Machine Learning in Metamaterial Design: Fragile Becomes Supercompressible.* Advanced Materials,
31(48), 1-6. [doi:10.1002/adma.201904845](https://doi.org/10.1002/adma.201904845)

**How:** We try to follow Murphy's book closely, but the sequence of Chapters and Sections is
different. The intention is to use notebooks as an introduction to the topic and Murphy's book
as a resource.
* If working offline: Go through this notebook and read the book.
* If attending class in person: listen to me (!) but also go through the notebook in your laptop at the same time. Read the book.
* If attending lectures remotely: listen to me (!) via Zoom and (ideally) use two screens where you have the notebook open in 1 screen and you see the lectures on the other. Read the book.

This is the second of two lectures on **adda**. Lecture 19.1 was how you build such a framework;
today is what happened when we pointed it at a real problem: the one that is also your final
project.

## **OPTION 1**. Run this notebook **locally in your computer**:
1. Confirm that you have the '3dasm' mamba (or conda) environment (see Lecture 1).
2. Go to the 3dasm_course folder in your computer and pull the last updates of the [repository](https://github.com/bessagroup/3dasm_course):
```
git pull
```
    - Note: if you can't pull the repo due to conflicts (and you can't handle these conflicts), use this command (with **caution**!) and your repo becomes the same as the one online:
```
git reset --hard origin/main
```
3. Open command window and load jupyter notebook (it will open in your internet browser):
```
jupyter notebook
```
5. Open notebook of this Lecture and choose the '3dasm' kernel.

## **OPTION 2**. Use **Google's Colab** (no installation required, but times out if idle):

1. go to https://colab.research.google.com
2. login
3. File > Open notebook
4. click on Github (no need to login or authorize anything)
5. paste the git link: https://github.com/bessagroup/3dasm_course
6. click search and then click on the notebook for this Lecture.

In [1]:
# Basic plotting tools needed in Python.

import matplotlib.pyplot as plt # import plotting tools to create figures
import numpy as np # import numpy to handle a lot of things!

%config InlineBackend.figure_format = "retina" # render higher resolution images in the notebook
plt.rcParams["figure.figsize"] = (8,4) # rescale figure size appropriately for slides

# To limit the number of rows to show in a dataframe, for presentation purposes:
import pandas as pd

pd.set_option('display.max_rows', 10)

## Outline for today

* The problem, which is also **your final project**
* What we asked for, and the clause that made it hard
* Four design ideas in detail, and a table of the rest
* **The headline it almost reported**, and how it caught itself
* Two ways the measurement lied
* The rules *we* changed while it was searching
* The honest scoreboard

**Reading material**: this notebook + Bessa, Glowacki & Houlder (2019).

Set expectations in the first minute. If students expect a triumphant "the AI discovered a new
metamaterial", they will misread everything that follows. The interesting result is not a winning
design. It is a map of a design space, produced quickly, with honest verdicts, including on ideas
we were personally attached to.

The most important sentence in the lecture is on the scoreboard slide: the best design found is
the one a human found first. Say it plainly and without embarrassment.

## Supercompressible metamaterials

A slender lattice (a "rocking mast") of three longerons joined by a top and bottom ring.

Compress it and it *coils* rather than breaking: the structure buckles into a stable, tightly
wound configuration, and springs back. *Fragile becomes supercompressible.*

The design problem: choose the geometry that maximizes the normalized critical buckling load,
among designs that are actually coilable.

## The baseline

<img src=./img/bessa_baseline_native.gif width=32% align='right'>

The reference point from Bessa, Glowacki & Houlder (2019):

* circular longeron cross-section, 3 longerons, 1 storey, circular rings
* `ratio_d = 0.02005`, `ratio_pitch = 0.25`, `ratio_top_diameter = 0.2505`
* $\sigma_{cr,nd} = 0.1306$ kPa/longeron, max local strain $= 0.0198$, fully reversible coiling

Every "$\times$ Bessa" figure in this lecture is a multiple of that number.

The animation is a native Abaqus/CAE viewer export of this exact design, re-solved for the study's
provenance deck. Colour is axial strain from the simulation's own field output, not a schematic.

One caveat if a sharp student asks: this baseline was later re-measured under a corrected oracle
as 0.1122 kPa. We get to that on the contract-change slide.

## You have seen these parameters before

Those three symbols, `ratio_top_diameter`, `ratio_pitch` and `ratio_d`, are the design variables of
**your final project** (Lecture 20).

The oracle is the same too: a linear buckling analysis to check coilability, then a Riks
post-buckling solve for the strain the design actually reaches.

So today is not a guest seminar about someone else's problem. It is a report on what an agentic
framework found in *your* design space, before you go and search it yourself.

## What counts as a valid design

Two feasibility criteria, from the original paper:

* **maximum compression strain** (`mcs`) $\geq$ **80%**: it must coil most of the way down
* **maximum local strain** (`mls`) $\leq$ **2%**: on *any* strain component, shear included

A design with a spectacular buckling load that violates either one is not an answer.

This is the same filter you applied in the Lecture 19.1 exercise, where the naive answer was
1300 times too large. Keep that in mind for the next 40 minutes.

## What we asked for

> Find a rocking-mast lattice design with maximum normalized critical buckling load among
> coilable topologies, surpassing Bessa et al. (2019), using topology parameters the original
> paper left unexplored.

Design space: 13-dimensional: 4 topology integers + 9 cross-section parameters.

And the clause that made it a research problem rather than an optimization exercise:
**novelty must be a new shape or arrangement, not a resized cross-section.**

Why that matters: Bayesian optimization already solves a 13-dimensional box, and you will do it
yourself for the final project. If all we wanted was the best point inside a fixed
parametrization, we would not need an agent.

The frontier is inventing the parametrization. So we asked for that instead.

This is the intellectual pivot of the project, and it came from the PI. Prescribing a fixed box and
asking the agent to beat a baseline by some percentage wastes the one thing an LLM uniquely brings:
it has read the literature and can propose a representation nobody wrote down for this problem.

The cost of that reframe is that "did it win" becomes much harder to answer, because a number can
now be real and still not count. That difficulty is the subject of most of this lecture.

## Six weeks, in numbers

| | |
| :-- | :-- |
| closed runs | 23 |
| genuinely new design ideas tested | 28 (D1 – D28) |
| period | 2026-06-29 → 2026-08-09 |
| cost per run | \$20.68 – \$54.45 |
| designs evaluated in the largest single campaign | 330 |

Every run ended in a critic-reviewed, reproduction-gated notebook, or it did not end.

## Beating the baseline was never the hard part

The floor fell early and kept falling. A rectangular longeron turned radially short and
tangentially long reached **5.9$\times$ Bessa** and became the study's own reference, `run17_rectangle`.
Pretwisted legs, a Kresling hinge and a leaf-spring longeron were all tried and all settled.

But every one of those is the same mast with a different cross-section. What we wanted was a
design that is new in *shape or arrangement*, and that is a different search.

Three attempts at that follow. Each begins with where the idea came from, because that is the
part a search cannot supply.

## D10 · Elliptical rings with a phase offset: *our* idea

<img src=./img/elliptical_rings_native.gif width=30% align='right'>

* **Where it came from:** no citation, and the record says so. A symmetry argument: circular rings
  force every longeron through the same peak curvature, so break the symmetry and the strain might
  redistribute.
* **What:** replace the circular rings with independently parametrized ellipses, plus a phase
  offset between their major axes, breaking the rotational symmetry that forces every longeron
  into identical peak curvature.
* **Stats:** n = 67 → 9 coilable → 9 Riks → **0 feasible**.
* **Verdict:** `INCONCLUSIVE · DEAD-END`

This was not the agent's idea. It was ours: the worked example in our own design-space framework,
the concrete illustration of what "invent a new parametrization" was supposed to mean.

The finding: `mcs` collapses from $0.9999$ to $0.398$ at the first non-circular step tested.
Elliptical rings sharply destroy coilability rather than redistributing strain.

The system tested our favourite idea and told us it did not work, with a funnel, a quartile table,
and a mechanism. That is the product: a fast, honest *no* on an idea we would otherwise have spent
a month on ourselves.

Do not rush this slide. It is the most persuasive argument in the lecture for the whole approach,
and it lands precisely because the failed idea was the PI's own and is documented as such in the
framework spec.

Note the epistemic care: INCONCLUSIVE rather than FALSIFIED, because the guiding constraint
surrogates were not demonstrably above chance, so a closed non-existence verdict is not licensed.
The raw picture is about as close to a clean dead end as the study ever sees, and the status still
refuses to overclaim.

## D21 · Tensegrity longerons: 1691$\times$ Bessa

<img src=./img/tensegrity_native.gif width=28% align='right'>

* **Where it came from:** Amendola et al. (2018) on tensegrity prestress stiffness, read against
  Meng (2012) and Sorrentino (2021) on how bending families couple strain to stiffness. If stiffness
  comes from prestress instead of bending, the strain penalty might vanish.
* **What:** replace the bending longeron with a pin-jointed, prestressed Class-1 tensegrity
  assembly: stiffness from prestress and geometry, not beam bending.
* **Origin:** Amendola et al. (2018), contrasted against Meng (2012) / Sorrentino (2021). A real
  citation.
* **Stats:** n = 45 → 45 coilable → 44 Riks → **12 feasible** (1691$\times$ Bessa).
  Best: $\sigma = 220.89$ kPa, mls $\approx 9\times10^{-14}$.

The largest $\sigma_{cr,nd}$ in the entire study. Re-verified by direct extraction from the
simulation output. Not an error.

**Verdict:** `SUPPORTED (DISQUALIFIED) · DEAD-END`

Look at the strain: mls $\approx 9\times10^{-14}$. The material is not straining *at all*. It
reaches 80% compression by rotating rigid bars about pin joints: a folding linkage, not elastic
coiling.

A real number measuring the wrong mechanism.

So we changed the rules: a design that reaches the compression target by rotating rigid bars, with
almost no material stretch, stopped counting. That was our decision, taken three weeks in, and it
is why the largest number in the study is not the answer.

1691x is the number a press release would have led with. It is also worthless as an answer to the
question we asked, and the system's own record says so in the same breath as reporting it.

This is also the clearest example of the human owning the contract. Nothing in the physics told us
folding linkages don't count: that is a judgement about what we are trying to build. The agent
found a legitimate exploit in an underspecified brief; we closed it.

## D25 · Tape-spring longerons: the biggest campaign

<img src=./img/tape_spring_native.gif width=27% align='right'>

A thin-walled open circular arc: the tape measure that snaps flat and coils.

* **Where it came from:** Calladine's inextensional shell-folding theory and Seffen and Pellegrino
  on tape-spring mechanics. A tape measure snaps flat and coils without stretching, which is exactly
  the behaviour the mast needs.
* **What was tested:** two campaigns, **330 designs** under contact, plus 10 paired
  contact-on/contact-off.
* **Result:** **0 feasible**, and the binding criterion is unanimous.

Of the 28 designs that reached a verdict: **28 fail on compression, 0 fail on strain.** Median
reaches 2.2% compression; the best reaches 21%, against the 80% required. Short by 3.7$\times$.

Of six free parameters, exactly one moves the blocker: the arc angle
($\rho = -0.600$, Holm-adjusted $p = 0.003$; every other parameter adjusts to $p = 1.00$).

And the best designs sit *on its lower bound.*

Normally "the optimum sits on a bound" means *widen the bound.* Here it means the opposite: the
search is pushing toward the shallowest arc allowed, and widening it further does not find a better
tape spring: it deletes the arc.

The best design in the campaign has 0.5 mm of section depth. It is a flat strip, which is D6
territory, already searched. **The optimizer's preferred direction exits the family.**

Worth separating two things you will meet in your own searches: *we looked hard and found nothing*
is a statement about your search. *Nothing in this family could clear the bar* is a statement about
the family. Only the second one closes a direction.

This is the best piece of reasoning in the study, and nobody taught it explicitly. "The optimizer
wants to leave the design space" is a qualitatively different and much stronger kind of evidence
than "we sampled a lot and found nothing".

Worth asking the room: would you have drawn the same conclusion from a parameter pinned at its
bound? Most people's trained reflex is to widen the bound.

## Six designs that appeared to peak late

In one run, six designs looked like they reached peak load near 50% compression, *after* coiling,
which would have been a genuinely new mechanism.

One of them reported $\sigma_{peak} = 245.8$ kPa: roughly **400$\times$ the reference design.**

This would have been the headline. It would have been on a slide, and very likely in a paper.

## What would you check?

Before reading on: you have a suspiciously large peak load from a Riks solve that stopped early.

**What single quantity would tell you whether the load really turned over?**

The number of increments *inside the measurement window*, against the number the solve actually
computed.

All six shared `window_n == history_n`: the reported peak was at the **last computed increment.**
The solve had died, and the final frame before it died was the largest load seen.

In [2]:
# Illustrative: these seven rows are typed out to show the diagnostic, not exported from the run.
# window_n  = increments inside the measurement window
# history_n = increments the solve actually computed
designs = pd.DataFrame({
    "design":     ["A", "B", "C", "D", "E", "F", "legit"],
    "sigma_peak": [245.8, 88.4, 41.2, 33.7, 22.9, 18.1, 0.6071],
    "window_n":   [61, 44, 52, 38, 70, 29, 70],
    "history_n":  [61, 44, 52, 38, 70, 29, 73],
})
designs["truncated"] = designs["window_n"] == designs["history_n"]
designs

,design,sigma_peak,window_n,history_n,truncated
0,A,245.8000,61,61,True
1,B,88.4000,44,44,True
2,C,41.2000,52,52,True
3,D,33.7000,38,38,True
4,E,22.9000,70,70,True
5,F,18.1000,29,29,True
6,legit,0.6071,70,73,False


The last row is the legitimate design: `window_n = 70 < history_n = 73`. The window closed *before*
the solve ran out, so the response really did turn over inside the measured range.

Three increments of difference separate a real result from a 400$\times$ artifact.

## Who caught it

The strategizer found that signature itself. It did not bank the number: it registered the
suspicion as a hypothesis, spent a delegation on a targeted falsification of its own would-be
headline, and reported the six as artifacts.

Read that as a claim about the framework rather than the model. The system was built so that
*"this number is too good"* is a testable hypothesis with somewhere to live, and so that closing it
required evidence.

Absent the hypothesis ledger, "huh, that's surprising" has nowhere to go except into the abstract.

This is the emotional centre of the lecture. Deliver it slowly.

Be precise about the credit. The model noticed an anomaly, and good models do that. What the framework
contributed is that noticing had consequences. A capable model with nowhere to put a doubt will
usually resolve the doubt in favour of the exciting answer, because that is what the surrounding
text rewards.

Also name honestly: the truncation convention that made this detectable was our engineering, added
because an earlier run had been fooled.

## A campaign printed this. What is wrong with it?

<br>

```
designs under the 2% strain limit: 36 of 36
closest miss: mls = 0.000000  mcs = 0.0000
```

<br>

Both lines are wrong, and each is wrong in a different way. Take a moment.

**Line 1 is a tautology.** The measurement window *closes at the 2% crossing*, so windowed `mls` is
bounded above by 0.02 by construction. Its maximum cannot exceed the limit it is being compared
against. Comparing it measures the ceiling, not the design. It carries no ranking information at
all.

**Line 2 ranks by `min(mls)`**, which picks the design that *stalled soonest*: the one that barely
strained because it barely did anything.

The unsaturated statistic is *at what compression does the design cross 2%*. Re-analysed on it, the
verdict changed.

This trap produced a wrong verdict twice, in two different disguises. Which is why it is written up
as a documented trap, not merely fixed in code.

## A statistic that cannot fail is not a measurement

You have met this before and you will meet it again:

* accuracy on a badly unbalanced dataset
* $R^2$ computed on the training set (Lecture 10)
* a p-value from a test whose assumptions were chosen after seeing the data

**Which of these have you reported this semester?**

## The rules moved while we were searching

Feasibility, the compression target, what counts as coiling: we changed all of them mid-study, more
than ten times. Two consequences you should carry:

* **Verdicts from different weeks are not comparable.** The same incumbent reads as
  5.9$\times$ under the old measurement and 5.42$\times$ under the corrected one.
* **The record is append-only.** A changed rule never rewrites an old verdict, it marks it for
  re-testing, because that verdict was a correct call under the rules of its own time.

## Where the human was indispensable

**Defining what counts.** Folding linkages, ring passthrough, the novelty clause. Nothing in the
physics says a prestressed pin-jointed linkage is not an answer: that is a judgement about what we
are trying to build.

**Fixing our own fidelity errors.** We removed ground contact believing it artificial. It was not.

**Stopping a false boundary from propagating.** A solve-time cap was described in the brief as
"a hard property". It was a cost argument, and a run closed early reasoning correctly from a premise
we had written wrongly.

**Writing down what must not be inherited**: that a mechanism ceiling established inside one
element type is not a boundary on the design space in general.

**Demanding the negative record.** One run tested a new family, falsified it, and its summary
pointed at "archived" because nobody had written the slide. A genuinely new family became invisible
to the next reader.

## The case for it as a partner

**Breadth we cannot match.** 28 distinct design families in six weeks, each with a stated origin,
real simulations, a funnel, and a verdict. Each would have been a graduate-student month.

**It killed its own headline.** A 400$\times$ result, caught by its own diagnostic, falsified by its
own delegation, reported as an artifact.

**It produced a mechanism, not a ranking.** One run fitted
$\sigma_{eig} \propto J^{0.96}$. Coil-mode critical load is set by the *minimum* of winding-plane
bending and torsional stiffness. Its own summary calls this *"the run's one durable contribution —
a mechanism law with an exponent, not a ranking."*

**It recorded its failures so they stay dead.** A family that was tried and settled is written down
as settled, so a later run does not spend a campaign rediscovering that leaves do not help.

## The honest scoreboard

The best feasible design after 23 runs and 28 design families:

## 0.6071 kPa
### the incumbent rectangle, *rediscovered*

found as the leaf-spring family's own regression control.

Under the corrected model the reference design measures $0.1122$ kPa rather than $0.1306$, and
the same incumbent re-measures at $0.6077$ kPa. That is **5.42$\times$ the reference**: a real,
large improvement on the published baseline.

And it is a design the search had already found weeks earlier, arrived at a second time through a
different code path.

**The novelty half of the objective was not met.** Twenty-seven other families were proposed,
motivated, simulated, and killed.

The value was never going to be a winning design in six weeks. The value is the **map**: 28
families with honest verdicts, and a record of which directions are dead so nobody walks them
again. A negative result that is trustworthy and recorded beats a positive one that is neither.

## Summary

* We asked for a **new shape or arrangement** that beats a published metamaterial, not a better
  point in a fixed box.
* Across 23 runs it proposed and tested 28 design families, most from real literature, and killed
  almost all of them with mechanisms attached.
* It **caught its own 400$\times$ headline** as a truncation artifact and falsified it.
* The measurement lied in more than one way; a statistic that cannot fail is not a measurement.
* **We changed the rules more than ten times while it searched.** The contract, the fidelity of
  the model, and what counts as an answer were ours throughout, and we got some of them wrong.
* The best design is still the one found early. The novelty objective was not met, and the system
  says so itself.

## Now it is your turn

You have the same problem, the same parameters, and the same two-stage oracle for your
**final project** (Lecture 20).

Two things from these two lectures are worth carrying into it:

1. Apply the feasibility filter *before* you look for a maximum. The naive answer in the
   Lecture 19.1 exercise was 1300$\times$ too large.
2. When your search stops improving, that is not evidence that nothing better exists. Say what
   power your search had.

### See you next class

Have fun!